# POC Extraction de données

Extraction des données provenant des rapport de G2K vers un format plus simple.

Import, définition des variables et récupération du contenue du rapport (sous forme de `string`)

In [1]:
import pandas as pd
import re

## Variable global
content = ""
sections_content = {}
data_report = {}
sections_titles = []

# Récuperaction du contenue du rapport
with open("../data/rapport-RGTh_3cm.txt", "r") as f:
    content = f.read()

Définition de mes *regex*

In [2]:
## Global pattern
section_header_pattern = re.compile(r"^\*+$\n^\*{5}(.*)\*{5}$\n^\*+$", re.MULTILINE)

## S1
s1_kv_pattern = re.compile(r"^([A-ZÀ-Ÿ][a-zA-ZÀ-ÿ' ]+?)\s*:\s*(.*)$", re.MULTILINE)

##S2

## S6
s6_nucleide_line_pattern = re.compile(
    r"^[+>?]\s+([A-Z]{1,2}-\d{1,3})\s+(\S+)\s+(\S+)\s+(\S+)\s+(\S+)\s+(\S+)\s+(\S+)\s+(\S+)\s+(\S+)",
    re.MULTILINE
)

## Extraction des header de chaque section
1 - On extrait les titre et on créer un dictionnaire de DataFrame

2- On découpe `content` par section que l'on range dans un dictionnaire `sections_content`

In [3]:
section_headers = section_header_pattern.findall(content)

# 1
for i, section in enumerate(section_headers):
    sections_titles.append(section.strip())
    data_report[section.strip()] = None

# 2
matches = list(section_header_pattern.finditer(content))
for i, match in enumerate(matches):
    title = match.group(1).strip()
    start = match.end()
    end = matches[i + 1].start() if i + 1 < len(matches) else len(content)
    sections_content[title] = content[start:end].strip()

### S1 - RAPPORT DE L’ANALYSE DU SPECTRE
C'est les metadata du rapport

In [4]:
s1_key = sections_titles[0]
matches = s1_kv_pattern.findall(sections_content[s1_key])
data_report[s1_key] = {key.strip(): value.strip() for key, value in matches}
data_report[s1_key]

{'Nom du fichier': 'C:\\GENIE2K\\CAMFILES\\SPECTRES\\STANDARDS\\Standard_Syrah_2024\\S',
 'Date du rapport': '02/06/2026 11:00:22',
 'Description': 'ech',
 'Identification': "Type d'échantillon          :",
 'Géométrie de mesure': 'Sensibilité de recherche    :  2.50',
 'Zone de recherche des pics': '50 - 16384',
 'Zone de calcul des surfaces': '50 - 16384',
 'Tolérance sur les énergies': '1.000 keV',
 'Date de mesure': '01/03/2024 12:08:24',
 'Temps actif': '253578.4 secondes',
 'Temps réel': '253655.5 secondes',
 'Temps mort': '0.03 %',
 'Nom de la géométrie': 'Rapport sur Les paramètres des pics  02/06/2026 11:00:22          Page  2'}

### S2 - RAPPORT ANALYSE DES PICS

### S3 - RAPPORT IDENTIFICATION DES NUCLEIDES

### S4 - RAPPORT IDENTIFICATION AVEC CORRECTION D’INTERFERENCE

### S5 - RAPPORT LIMITES DE DETECTION

### S6 - RAPPORT LIMITES DE DETECTION  ISO 11929

In [39]:
def extract_header_s6(sections_content):
    header = []
    str = sections_content[sections_titles[5]]

    pattern = re.compile(r"^\s+(.*)$\n^\s+(.*)$", re.MULTILINE)
    lignes = pattern.findall(str)

    l1 = re.findall(r"([A-Za-zÀ-ÿ]+\.?)", lignes[0][0])
    l2 = re.findall(r"([A-Za-zÀ-ÿ]+\.?)", lignes[0][1])
    # Fusion en partant de la fin
    for a, b in zip(reversed(l1), reversed(l2)):
        header.insert(0, f"{a} {b}".strip())

    # Si l1 est plus longue, on conserve le début restant
    if len(l1) > len(l2):
        header = l1[:len(l1) - len(l2)] + header

    return header

header = extract_header_s6(sections_content)

lignes = re.findall(s6_nucleide_line_pattern, sections_content[sections_titles[5]])

data_report["RAPPORT LIMITES DE DETECTION  ISO 11929"] = pd.DataFrame(lignes, columns=header)
df = data_report["RAPPORT LIMITES DE DETECTION  ISO 11929"]

df[header[1:]] = df[header[1:]].apply(pd.to_numeric, errors="coerce")

print(df)


  Nucléide      LD      SD  Limite Basse  Limite Haute  Moyenne Activité  \
0   TL-208   12.00    5.87        682.00        719.00            701.00   
1   PB-210   29.40   14.60         40.60         60.30             50.50   
2   BI-212   76.60   38.00       2980.00       3400.00           3190.00   
3   BI-214    9.56    4.65         56.00         66.80             61.40   
4   PB-214    4.91    2.44         58.70         62.80             60.80   
5   AC-228   14.40    7.12       1500.00       1550.00           1520.00   
6   TH-228    3.02    1.49       2050.00       2100.00           2080.00   
7   TH-230  381.00  186.00        329.00        601.00            465.00   
8    U-235    2.16    1.07          7.81          9.49              8.65   
9   AM-241    4.38    2.06          3.00          6.74              4.87   

   pondérée Incert.  Meilleure Activité  Estimation Incert.  
0             9.270              701.00              18.500  
1             5.030               50.50

# Zone de Test

In [ ]:


extract_header_s6(sections_content)






['Nucléide',
 'LD',
 'SD',
 'Limite Basse',
 'Limite Haute',
 'Moyenne Activité',
 'pondérée Incert.',
 'Meilleure Activité',
 'Estimation Incert.']